# Load library

In [1]:
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn as skl
import anndata as ann
import random, os
from scipy.stats import pearsonr as pr
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import roc_auc_score
from sklearn.metrics import accuracy_score
from sklearn.metrics import f1_score as f1
from sklearn.metrics import precision_recall_curve as prc
from sklearn.metrics import silhouette_score as sil
from sklearn.metrics import auc
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_score, recall_score, average_precision_score
from sklearn.metrics import silhouette_score
from torch_geometric.nn import TransformerConv
from torch_geometric.data import Data
import psutil
import os, sys
import gc
import scipy.sparse as sp
from harmony import harmonize
from tqdm import tqdm
import h5py

In [2]:
sc.set_figure_params(dpi=200)

# General processing functinos

In [3]:
def whats_memory_eater():
    # Build reverse map of object id -> variable name from globals
    name_map = {id(obj): name for name, obj in globals().items()}

    # Get all tracked objects
    all_objects = gc.get_objects()

    # Safely get size and match variable name
    sizes = []
    for obj in all_objects:
        try:
            size = sys.getsizeof(obj)
            obj_id = id(obj)
            name = name_map.get(obj_id, None)
            sizes.append((size, type(obj), name, repr(obj)[:100]))
        except Exception:
            continue

    # Sort and print top 10
    sizes.sort(reverse=True, key=lambda x: x[0])

    for size, obj_type, name, preview in sizes[:10]:
        print(f"Size: {size / 1024**3} GB | Type: {obj_type} | Name: {name} | Object: {preview}")


In [4]:
def memory_usgae():
    gc.collect()
    process = psutil.Process(os.getpid())
    memory_gb = process.memory_info().rss / 1024**3  # in GB

    print(f"Current memory usage: {memory_gb:.2f} GB")

In [5]:
def load_and_preprocess_project(base_path, Project_ID, metadata_idx_key='Cell', 
                                Primary_or_Metastatic = 'Primary', remove_doublets = True, doublet_rate=0.06,
                                further_pre = False, file_prefix= None):
    """
    Load and preprocess a single scRNA-seq project with standard filtering and UMAP.
    
    Assumes the base_path contains:
        - One .mtx file (count matrix)
        - One barcodes.csv
        - One features.csv
        - One meta_all.csv
    """
    # Automatically detect files
    files = os.listdir(base_path)
    metadata_file = None
    
    if file_prefix == None:

        mtx_file = [os.path.join(base_path, f) for f in files if f.endswith('.mtx')][0]
        print(mtx_file)
        barcodes_file = [os.path.join(base_path, f) for f in files if 'barcode' in f][0]
        print(barcodes_file)
        try:
            features_file = [os.path.join(base_path, f) for f in files if 'feature' in f][0]
        except:
            features_file = [os.path.join(base_path, f) for f in files if 'genes' in f][0]
        print(features_file)
        metadata_file = [os.path.join(base_path, f) for f in files if 'meta' in f][0]
        print(metadata_file)
    else:
        for f in files:
            if not f.startswith(file_prefix):
                continue
            if f.endswith('mtx'):
                mtx_file = os.path.join(base_path, f)
                print(mtx_file)
            elif 'barcode' in f:
                barcodes_file = os.path.join(base_path, f)
                print(barcodes_file)
            elif 'feature' in f:
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'genes' in f:
                features_file = os.path.join(base_path, f)
                print(features_file)
            elif 'meta' in f:
                metadata_file = os.path.join(base_path, f)
                print(metadata_file)
            else:
                continue
    # print(metadata_file)
    print(f"Loading: {mtx_file}")

    # Load matrix
    adata = sc.read_mtx(mtx_file)
    adata = adata.transpose()  # Important: make cells as rows, genes as columns

    # Load barcodes and features
    if barcodes_file.endswith('tsv'):
        barcodes = pd.read_csv(barcodes_file, sep='\t', header=None)  # no header=None here
    else:
        barcodes = pd.read_csv(barcodes_file)  # no header=None here
    display(barcodes)
    
    if features_file.endswith('tsv'):
        genes = pd.read_csv(features_file, sep='\t', header=None)  # no header=None here
    else:
        genes = pd.read_csv(features_file)  # no header=None here
    display(genes)

    # Assign barcodes and gene names (convert to string)
    if barcodes.shape[1] > 1:
        adata.obs_names = barcodes.iloc[:, 1].astype(str).values
    else:
        adata.obs_names = barcodes.iloc[:, 0].astype(str).values
    
    if genes.shape[1] > 1:
        adata.var_names = genes.iloc[:, 1].astype(str).values
    else:
        adata.var_names = genes.iloc[:, 0].astype(str).values
    # adata.var_names = genes.iloc[:, 0].astype(str).values
    display(adata.to_df())

    # Load and merge metadata
    # if metadata_file
    try:
        if metadata_file.endswith('tsv'):
            metadata = pd.read_csv(metadata_file, sep='\t')
        elif metadata_file.endswith('csv'):
            metadata = pd.read_csv(metadata_file)
        metadata.index = metadata[metadata_idx_key]
        adata.obs = adata.obs.join(metadata, how='left')
    except:
        pass         
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')

    # Calculate QC metrics
    adata.var['mt'] = adata.var_names.str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

    # Standard cell filtering
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) & 
                  (adata.obs['n_genes_by_counts'] <= 5000) & 
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    # Normalize and log transform
    adata.raw = adata.copy()
    
    if further_pre:
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # Highly variable genes
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)

        # Keep only HVGs
        # adata = adata[:, adata.var.highly_variable]

        # Scale
        sc.pp.scale(adata, max_value=10)

        # PCA
        sc.tl.pca(adata, svd_solver='arpack')

        # Neighbors
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)

        # UMAP
        sc.tl.umap(adata)

    print(f"Finished processing {Project_ID}. Shape: {adata.shape}")

    return adata


In [6]:
def filter_and_recompute(adata, celltype_col, celltypes_to_keep, further_pre = False):
    """
    Filters an AnnData object to keep only specified cell types, 
    then recalculates PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object
    - celltype_col: str, the column in adata.obs containing cell type annotations
    - celltypes_to_keep: list of str, the cell types you want to keep

    Returns:
    - filtered and recalculated AnnData object
    """
    # Step 1: Filter cells
    print(f"Original shape: {adata.shape}")
    adata_filtered = adata[adata.obs[celltype_col].isin(set(celltypes_to_keep))].copy()
    print(f"Filtered shape: {adata_filtered.shape}")

    # Step 2: Recalculate PCA and UMAP
    # (Assumes data is already normalized and scaled)
    if further_pre:
        sc.tl.pca(adata_filtered, svd_solver='arpack')
        sc.pp.neighbors(adata_filtered, n_neighbors=15, n_pcs=40)
        sc.tl.umap(adata_filtered)

    print("Recalculated PCA and UMAP.")
    return adata_filtered

In [32]:
def reprocess_from_raw_layer(adata, Project_ID, Primary_or_Metastatic='Primary', 
                             further_pre=False, remove_doublets=True, doublet_rate=0.06):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes doublet removal, normalization, HVG selection, PCA, neighbors, and UMAP.
    
    Parameters:
    - adata: AnnData object, must have .raw set
    - Project_ID: Project identifier
    - Primary_or_Metastatic: Sample type ('Primary' or 'Metastatic')
    - further_pre: Whether to do full preprocessing (normalization, PCA, UMAP)
    - remove_doublets: Whether to run doublet detection and filtering
    - doublet_rate: Expected doublet rate (default 0.06 = 6%)
    
    Returns:
    - Processed AnnData object (modifies in place)
    """
    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")
    
    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()
    
    # DOUBLET DETECTION (before other filtering)
    if remove_doublets:
        print(f'Running doublet detection on {adata.n_obs} cells...')
                
        sc.external.pp.scrublet(adata, expected_doublet_rate=0.06)
        
        n_cells = adata.n_obs
        n_doublets = adata.obs['predicted_doublet'].sum()
        print(f'  Detected {n_doublets} doublets ({n_doublets/n_cells*100:.1f}%)')
        
        adata = adata[~adata.obs['predicted_doublet']].copy()
        print(f'  After doublet removal: {adata.n_obs} cells')
    
    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # Add metadata
    adata.obs['Project_ID'] = Project_ID
    adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        print('Normalizing...')
        # Normalize and log transform
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)
        
        print('Scaling...')
        sc.pp.scale(adata, max_value=10)
        
        print('Computing PCA...')
        sc.tl.pca(adata, svd_solver='arpack')
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)
    
    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata

In [8]:
def reprocess_all(adata, further_pre = True):
    """
    Reprocess a Scanpy AnnData object using its raw layer (e.g., from a published .h5ad).
    This includes normalization, HVG selection, PCA, neighbors, and UMAP.

    Parameters:
    - adata: AnnData object, must have .raw set

    Returns:
    - Processed AnnData object (modifies in place)
    """

    # Check if raw exists
    if adata.raw is None:
        raise ValueError("AnnData object has no .raw attribute. Cannot proceed with reprocessing.")

    # Extract raw counts
    adata.X = adata.raw.X.copy()
    adata.var = adata.raw.var.copy()
    adata.var_names = adata.raw.var_names.copy()

    # Recalculate mitochondrial content
    adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)
    
    print('Standard filtering...')
    # Standard filtering (optional)
    adata = adata[(adata.obs['n_genes_by_counts'] >= 200) &
                  (adata.obs['n_genes_by_counts'] <= 5000) &
                  (adata.obs['pct_counts_mt'] <= 20)].copy()
    
    # adata.obs['Project_ID'] = Project_ID
    # adata.obs['Primary_or_Metastatic'] = Primary_or_Metastatic
    
    if further_pre:
        # Normalize and log transform
        print('Normalizing...')
        sc.pp.normalize_total(adata, target_sum=1e4)
        sc.pp.log1p(adata)

        # HVG selection
        # sc.pp.highly_variable_genes(adata, min_mean=0.0125, max_mean=3, min_disp=0.5)
        # adata = adata[:, adata.var.highly_variable]
        '''
        sc.pp.highly_variable_genes(
            adata,
            flavor="seurat_v3",  # best for batch-aware HVG selection
            n_top_genes=2000,
            batch_key="Final_sample_id"  # or whatever your batch label column is
        )
        '''
        
        # adata = adata[:, adata.var.highly_variable].copy()

        # Scale
        # print('Scaling...')
        # sc.pp.scale(adata, max_value=10)
        print('Computing PCA...')
        # PCA, neighbors, UMAP
        sc.tl.pca(adata, zero_center=False)
        
        print('Computing neighbors...')
        sc.pp.neighbors(adata, n_neighbors=15, n_pcs=40)
        
        print('Computing UMAP...')
        sc.tl.umap(adata)

    print(f"Reprocessed dataset. Final shape: {adata.shape}")
    return adata


In [9]:
def reset_plot():
    # After running scrublet, reset the display settings
    import matplotlib.pyplot as plt
    import matplotlib
    %matplotlib inline

    # Reset matplotlib backend
    matplotlib.use('module://matplotlib_inline.backend_inline')

    # Reset scanpy settings
    import scanpy as sc
    sc.settings.autoshow = True
    sc.settings.set_figure_params(dpi=100, facecolor='white')

# Lung Cancer (LUCA)

## High-resolution single-cell atlas reveals diversity and plasticity of tissue-resident neutrophils in non-small cell lung cancer

Paper: https://www.sciencedirect.com/science/article/pii/S1535610822004998?via%3Dihub

Data downloaded from: https://cellxgene.cziscience.com/collections/edb893ee-4066-4128-9aec-5eb2b03f8287

Link: 
- Anndata (extended): https://datasets.cellxgene.cziscience.com/80b57568-2621-4911-b4b1-4f2cf5087962.h5ad
- Anndata (core): https://datasets.cellxgene.cziscience.com/f27535dc-7902-456d-b94f-024dfe8791c0.h5ad


In [10]:
memory_usgae()

Current memory usage: 0.63 GB


In [11]:
ad = sc.read_h5ad('../../Data/LUAD/High-resolution_single-cell_atlas.extended.h5ad')
ad

AnnData object with n_obs × n_vars = 1283972 × 17797
    obs: 'sample', 'uicc_stage', 'ever_smoker', 'age', 'donor_id', 'origin', 'dataset', 'ann_fine', 'cell_type_predicted', 'doublet_status', 'leiden', 'n_genes_by_counts', 'total_counts', 'total_counts_mito', 'pct_counts_mito', 'ann_coarse', 'cell_type_tumor', 'tumor_stage', 'TP53_mutation', 'ALK_mutation', 'BRAF_mutation', 'ERBB2_mutation', 'KRAS_mutation', 'ROS_mutation', 'origin_fine', 'study', 'platform', 'cell_type_major', 'cell_type_neutro', 'cell_type_neutro_coarse', 'suspension_type', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'tissue_type', 'EGFR_mutation', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'is_highly_v

In [12]:
ad.obs.cell_type.value_counts()

cell_type
CD4-positive, alpha-beta T cell         199444
CD8-positive, alpha-beta T cell         160394
alveolar macrophage                     147293
macrophage                              114381
malignant cell                           91528
classical monocyte                       86641
natural killer cell                      74816
B cell                                   69007
pulmonary alveolar type 2 cell           47999
regulatory T cell                        40051
plasma cell                              35538
CD1c-positive myeloid dendritic cell     28882
epithelial cell of lung                  28763
vein endothelial cell                    20843
mast cell                                19838
neutrophil                               19368
capillary endothelial cell               16898
multiciliated epithelial cell            12205
fibroblast of lung                       10292
non-classical monocyte                    9189
myeloid cell                              7663
pul

In [13]:
ad = ad[ad.obs.disease!= 'normal']
ad

View of AnnData object with n_obs × n_vars = 1071083 × 17797
    obs: 'sample', 'uicc_stage', 'ever_smoker', 'age', 'donor_id', 'origin', 'dataset', 'ann_fine', 'cell_type_predicted', 'doublet_status', 'leiden', 'n_genes_by_counts', 'total_counts', 'total_counts_mito', 'pct_counts_mito', 'ann_coarse', 'cell_type_tumor', 'tumor_stage', 'TP53_mutation', 'ALK_mutation', 'BRAF_mutation', 'ERBB2_mutation', 'KRAS_mutation', 'ROS_mutation', 'origin_fine', 'study', 'platform', 'cell_type_major', 'cell_type_neutro', 'cell_type_neutro_coarse', 'suspension_type', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'tissue_type', 'EGFR_mutation', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'is_

In [14]:
ad = filter_and_recompute(ad, 
                          celltype_col='cell_type', 
                          celltypes_to_keep=['malignant cell'],
                          further_pre=False)
ad

Original shape: (1071083, 17797)
Filtered shape: (90155, 17797)
Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 90155 × 17797
    obs: 'sample', 'uicc_stage', 'ever_smoker', 'age', 'donor_id', 'origin', 'dataset', 'ann_fine', 'cell_type_predicted', 'doublet_status', 'leiden', 'n_genes_by_counts', 'total_counts', 'total_counts_mito', 'pct_counts_mito', 'ann_coarse', 'cell_type_tumor', 'tumor_stage', 'TP53_mutation', 'ALK_mutation', 'BRAF_mutation', 'ERBB2_mutation', 'KRAS_mutation', 'ROS_mutation', 'origin_fine', 'study', 'platform', 'cell_type_major', 'cell_type_neutro', 'cell_type_neutro_coarse', 'suspension_type', 'assay_ontology_term_id', 'cell_type_ontology_term_id', 'development_stage_ontology_term_id', 'disease_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'is_primary_data', 'organism_ontology_term_id', 'sex_ontology_term_id', 'tissue_ontology_term_id', 'tissue_type', 'EGFR_mutation', 'cell_type', 'assay', 'disease', 'organism', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'is_highly_var

In [15]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='High-resolution_single-cell_atlas', 
                              Primary_or_Metastatic='Primary',
                              further_pre=True)

Running doublet detection on 90155 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.75
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 3.3%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.0%
  Detected 1 doublets (0.0%)
  After doublet removal: 90154 cells
Standard filtering...
Normalizing...
Scaling...
Computing PCA...
Computing neighbors...


/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1063: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1071: NumbaDeprecationWarning: The 'nopython' keyword argument was not supplied to the 'numba.jit' decorator. The implicit default value for this argument is currently False, but it will be changed to True in Numba 0.59.0. See https://numba.readthedocs.io/en/stable/reference/deprecation.html#deprecation-of-object-mode-fall-back-behaviour-when-using-jit for details.
  @numba.jit()
/home/wang3712/.local/lib/python3.8/site-packages/umap/distances.py:1086: NumbaDeprecationWarning:

Computing UMAP...
Reprocessed dataset. Final shape: (86472, 17797)


In [16]:
# sc.pl.umap(ad, color='is_primary_data')
sc.pl.umap(ad, color='disease')

sc.pl.umap(ad, color='tissue')
sc.pl.umap(ad, color='cell_type_tumor')
sc.pl.umap(ad, color='study')

/home/wang3712/.local/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/wang3712/.local/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/wang3712/.local/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(
/home/wang3712/.local/lib/python3.8/site-packages/scanpy/plotting/_tools/scatterplots.py:392: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [17]:
ad = ad[ad.obs.study != 'Wu_Zhou_2021']

In [18]:
primary_or_metastatic = []

for _, row in ad.obs.iterrows():
    if row['tissue'] != 'lung':
        primary_or_metastatic.append('Metastatic')
    elif 'IV' in str(row['uicc_stage']):
        primary_or_metastatic.append('Metastatic')
    elif row['uicc_stage'] == 'III':
        primary_or_metastatic.append('Locally advanced')
    else:
        primary_or_metastatic.append('Primary')

ad.obs['Primary_or_Metastatic'] = primary_or_metastatic


/tmp/ipykernel_1400833/1482568291.py:13: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  ad.obs['Primary_or_Metastatic'] = primary_or_metastatic


In [19]:
def assign_lung_subtype(row):
    if row["EGFR_mutation"] == "mutated":
        return "LUCA: EGFR-mutant"
    elif row["KRAS_mutation"] == "mutated":
        return "LUCA: KRAS-mutant"
    elif row["ALK_mutation"] == "mutated":
        return "LUCA: ALK-rearranged"
    elif row["ROS_mutation"] == "mutated":
        return "LUCA: ROS1-rearranged"
    elif row["BRAF_mutation"] == "mutated":
        return "LUCA: BRAF-mutant"
    elif row["ERBB2_mutation"] == "mutated":
        return "LUCA: ERBB2-mutant"
    elif row["TP53_mutation"] == "mutated":
        return "LUCA: TP53-mutant (no driver)"
    else:
        return "LUCA: Unspecified"

In [20]:
ad.obs['Project_ID'] = ad.obs['study']

ad.obs['Final_cancer_type']  = 'Lung Cancer'
ad.obs['Final_histological_subtype'] = [i[2] for i in ad.obs['cell_type_tumor'].str.split(' ')]
ad.obs['Final_molecular_subtype'] = ad.obs.apply(assign_lung_subtype, axis=1)
ad.obs['Final_tissue'] = ad.obs['tissue'].astype(str).str.capitalize()
ad.obs['Final_sample_id'] = ad.obs.donor_id

In [21]:
ad.obs['development_stage']

052CO-a_GGTGAAGTCCCGGATG-0    62-year-old stage
056CO_CTTCTCTTCCTGCAGG-0      57-year-old stage
056CO_GTCTCGTCATTCCTCG-0      57-year-old stage
056CO_TTGCGTCCATGAACCT-0      57-year-old stage
137CO_AACTGGTCACCGAATT-0      73-year-old stage
                                    ...        
TGGCCAGCACGTAAGG-1-34-8       68-year-old stage
TGGCCAGGTCATCCCT-1-34-8       68-year-old stage
TTATGCTAGAGAGCTC-1-34-8       68-year-old stage
TTTCCTCTCCTTGGTC-1-34-8       68-year-old stage
GCAGCCAAGATCTGCT-1-35-8       76-year-old stage
Name: development_stage, Length: 54027, dtype: category
Categories (38, object): ['36-year-old stage', '37-year-old stage', '41-year-old stage', '43-year-old stage', ..., '83-year-old stage', '86-year-old stage', '90-year-old stage', 'unknown']

In [22]:
import numpy as np

def parse_age(val):
    val = str(val).strip().lower()
    if val == "unknown" or val == "":
        return np.nan
    elif "-" in val:
        try:
            return int(val.split("-")[0])
        except:
            return np.nan
    else:
        try:
            return int(val)
        except:
            return np.nan

ad.obs['Final_patient_age'] = ad.obs['development_stage'].apply(parse_age)


In [23]:
# add more clinical information
ad.obs['Final_patient_stage'] = ad.obs['uicc_stage']
# ad.obs['Final_patient_treatment'] = ad.obs['Treatment most recent class']+' '+ad.obs['Treatment status']

In [24]:
ad.obs['Project_ID'].value_counts()

Project_ID
Kim_Lee_2020                 16389
Chen_Zhang_2020              10390
Lambrechts_Thienpont_2018     6744
UKIM-V                        5377
He_Fan_2021                   4681
Leader_Merad_2021             4335
Zilionis_Klein_2019           1986
Laughney_Massague_2020        1542
Goveia_Carmeliet_2020         1468
Maynard_Bivona_2020            979
Adams_Kaminski_2020            136
Name: count, dtype: int64

In [25]:
# add more clinical information

ad.obs['Final_patient_treatment'] = 'Naïve'

In [26]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed/High-resolution_single-cell_atlas.LUCA.h5ad', compression='gzip')

## Signatures of plasticity, metastasis, and immunosuppression in an atlas of human small cell lung cancer

Paper: https://www.sciencedirect.com/science/article/pii/S1535610821004979?via%3Dihub

Data downloaded from: https://cellxgene.cziscience.com/collections/62e8f058-9c37-48bc-9200-e767f318a8ec

Link: 
- Anndata: https://datasets.cellxgene.cziscience.com/b9175586-5875-4346-afa2-5f23e15fe16d.h5ad

In [28]:
ad = sc.read_h5ad('../../Data/LUAD/HTAN_MSK_SCLC.h5ad')

In [29]:
ad

AnnData object with n_obs × n_vars = 147137 × 22397
    obs: 'ngenes', 'libsize', 'mito_frac', 'RBP_frac', 'batch', 'donor_id', 'treatment', 'procedure', 'histo', 'cell_type_coarse', 'cell_type_fine', 'cell_type_general', 'clusters', 'cell_type_med', 'H_knn', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'suspension_type', 'HTAN_Biospecimen_ID', 'HTAN_Participant_ID', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid'
    var: 'feature_is_filtered', 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type'
    uns: 'citation', 'default_embedding', 'organism', 'organism_ontology_term_id', 'schema_reference', 'schema_version', 'title'
    obsm: 'X_pca', 'X_umap'

In [33]:
# re-process the adata
ad = reprocess_from_raw_layer(ad, 
                              Project_ID='HTAN_MSK_SCLC', 
                              Primary_or_Metastatic='Primary',
                              further_pre=False)

Running doublet detection on 147137 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.66
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 13.4%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.5%
  Detected 106 doublets (0.1%)
  After doublet removal: 147031 cells
Standard filtering...
Reprocessed dataset. Final shape: (134505, 22397)


In [ ]:
reset_plot()
for obs in ['batch', 'cell_type_coarse', 'cell_type_fine', 'cell_type_general']:
    sc.pl.umap(ad, color=obs)

In [36]:
ad = filter_and_recompute(ad, 
                          celltype_col='cell_type_fine', 
                          celltypes_to_keep=['SCLC-A', 'SCLC-N', 'SCLC-P'],
                          further_pre=True)
ad

Original shape: (134505, 22397)
Filtered shape: (48221, 22397)
Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 48221 × 22397
    obs: 'ngenes', 'libsize', 'mito_frac', 'RBP_frac', 'batch', 'donor_id', 'treatment', 'procedure', 'histo', 'cell_type_coarse', 'cell_type_fine', 'cell_type_general', 'clusters', 'cell_type_med', 'H_knn', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'suspension_type', 'HTAN_Biospecimen_ID', 'HTAN_Participant_ID', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'feature_name', 'feature_reference', 'feature_biotype', 'feature_length', 'feature_type', 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by

In [39]:
patient_clinical_df = pd.read_csv('../../Data/LUAD/HTAN_MSK_SCLC.clinical.txt', sep='\t')
# patient_clinical_df = patient_clinical_df[patient_clinical_df['Profiling method'] == 'scRNAseq']
patient_clinical_df['Lab ID'] = patient_clinical_df['Lab ID'].str.upper()
patient_clinical_df.head()

,Lab ID,Data Source,Collection Site Code,Gender,Ethnicity,Race,Smoking Status,Pack Years,Vital Status,Stage at Dx,...,Pre-collection Therapies,Time between last drug administration and tissue collection (months),IO line of therapy,Pre-Tissue IO Type,Clinical ASCL1,Clinical NeuroD1,ASCL1,NeuroD1,YAP1,POU2F3
0,RU325,Current manuscript,PL,Female,Non-Spanish; Non-Hispanic ...,White,Never,0.0,Deceased,IV,...,carboplatin+etoposide; ipilimumab+nivolumab; t...,0.2,2L,combination ICB,NaN,NaN,NaN,NaN,NaN,NaN
1,RU1038,Current manuscript,T,Female,Non-Spanish; Non-Hispanic ...,White,Never,0.0,Alive,IA,...,NaN,NaN,na,na,NaN,NaN,NaN,NaN,NaN,NaN
2,RU1057,Current manuscript,T,Female,Non-Spanish; Non-Hispanic ...,White,Former,30.0,Alive,IIB,...,NaN,NaN,na,na,NaN,NaN,NaN,NaN,NaN,NaN
3,RU1061,Current manuscript,T,Female,Non-Spanish; Non-Hispanic ...,White,Current,37.5,Alive,IV,...,NaN,NaN,na,na,NaN,NaN,NaN,NaN,NaN,NaN
4,RU1065C,Current manuscript,LI,Female,Non-Spanish; Non-Hispanic,White,Current,20.0,Deceased,IV,...,carboplatin+etoposide+atezolizumab,0.7,1L,combination chemo-IO,NaN,NaN,pos,neg,neg,neg


In [40]:
ad.obs['donor_id'] = ad.obs['donor_id'].str.upper()

In [41]:
merged = ad.obs.reset_index().merge(
    patient_clinical_df,
    left_on='donor_id',
    right_on='Lab ID',
    how='left'
)

# Step 2: Restore original index (cell barcodes)
merged = merged.set_index('Cell')

# Step 3: Assign back
ad.obs = merged
ad

AnnData object with n_obs × n_vars = 48221 × 22397
    obs: 'ngenes', 'libsize', 'mito_frac', 'RBP_frac', 'batch', 'donor_id', 'treatment', 'procedure', 'histo', 'cell_type_coarse', 'cell_type_fine', 'cell_type_general', 'clusters', 'cell_type_med', 'H_knn', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'suspension_type', 'HTAN_Biospecimen_ID', 'HTAN_Participant_ID', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'Lab ID', 'Data Source', 'Collection Site Code', 'Gender', 'Ethnicity', 'Race', 'Smoking Status', 'Pack Years', 'Vital Status', 'Stage at Dx', 'Overall Sur

In [42]:
primary_or_metastatic = []
for i in ad.obs.tissue:
    if i != 'lung':
        primary_or_metastatic.append('Metastatic')
    else:
        primary_or_metastatic.append('Primary')
ad.obs['Primary_or_Metastatic'] = primary_or_metastatic

ad.obs['Project_ID'] = 'HTAN_MSK_SCLC'

ad.obs['Final_cancer_type']  = 'Lung Cancer'
ad.obs['Final_histological_subtype'] = 'SCLC'
ad.obs['Final_molecular_subtype'] = 'LUCA: Unspecified'
ad.obs['Final_tissue'] = ad.obs['tissue'].astype(str).str.capitalize()
ad.obs['Final_sample_id'] = ad.obs.donor_id

ad

AnnData object with n_obs × n_vars = 48221 × 22397
    obs: 'ngenes', 'libsize', 'mito_frac', 'RBP_frac', 'batch', 'donor_id', 'treatment', 'procedure', 'histo', 'cell_type_coarse', 'cell_type_fine', 'cell_type_general', 'clusters', 'cell_type_med', 'H_knn', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'suspension_type', 'HTAN_Biospecimen_ID', 'HTAN_Participant_ID', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'Lab ID', 'Data Source', 'Collection Site Code', 'Gender', 'Ethnicity', 'Race', 'Smoking Status', 'Pack Years', 'Vital Status', 'Stage at Dx', 'Overall Sur

In [43]:
def parse_age(val):
    val = str(val).strip().lower()
    if val == "unknown" or val == "":
        return np.nan
    elif "-" in val:
        try:
            return int(val.split("-")[0])
        except:
            return np.nan
    else:
        try:
            return int(val)
        except:
            return np.nan

ad.obs['Final_patient_age'] = ad.obs['development_stage'].apply(parse_age)


In [44]:
# add more clinical information
# ad.obs['Final_patient_age'] = 'Unknown'
ad.obs['Final_patient_stage'] = ad.obs['Stage at Dx']
ad.obs['Final_patient_treatment'] = ad.obs['treatment']

In [45]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed//Signature_atlas_SCLC.LUCA.h5ad', compression='gzip')

In [46]:
ad

AnnData object with n_obs × n_vars = 48221 × 22397
    obs: 'ngenes', 'libsize', 'mito_frac', 'RBP_frac', 'batch', 'donor_id', 'treatment', 'procedure', 'histo', 'cell_type_coarse', 'cell_type_fine', 'cell_type_general', 'clusters', 'cell_type_med', 'H_knn', 'development_stage_ontology_term_id', 'sex_ontology_term_id', 'self_reported_ethnicity_ontology_term_id', 'assay_ontology_term_id', 'is_primary_data', 'tissue_ontology_term_id', 'disease_ontology_term_id', 'cell_type_ontology_term_id', 'suspension_type', 'HTAN_Biospecimen_ID', 'HTAN_Participant_ID', 'tissue_type', 'cell_type', 'assay', 'disease', 'sex', 'tissue', 'self_reported_ethnicity', 'development_stage', 'observation_joinid', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'Lab ID', 'Data Source', 'Collection Site Code', 'Gender', 'Ethnicity', 'Race', 'Smoking Status', 'Pack Years', 'Vital Status', 'Stage at Dx', 'Overall Sur

## A pan-cancer blueprint of the heterogeneous tumor microenvironment revealed by single-cell profiling


Paper: https://www.nature.com/articles/s41422-020-0355-0#Fig3

Data downloaded from: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0

Link: 
- Matrix: https://lambrechtslab.sites.vib.be/en/pan-cancer-blueprint-tumour-microenvironment-0 (Lung cancer - Counts Matrix)
- Patient metadata: https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM13_ESM.pdf
- Sequecing quality: https://www.nature.com/
https://static-content.springer.com/esm/art%3A10.1038%2Fs41422-020-0355-0/MediaObjects/41422_2020_355_MOESM14_ESM.pdf

In [47]:
# Set the project directory
project_dir = "../../Data/LUAD/2096-Lungcancer/"  # <- change this for each project

# Load and preprocess
ad = load_and_preprocess_project(project_dir, 
                                 Project_ID='2096-Lungcancer', 
                                 Primary_or_Metastatic='Primary',
                                 further_pre=True)

../../Data/LUAD/2096-Lungcancer/matrix.mtx
../../Data/LUAD/2096-Lungcancer/barcodes.tsv
../../Data/LUAD/2096-Lungcancer/genes.tsv
../../Data/LUAD/2096-Lungcancer/2097-Lungcancer_metadata.csv
Loading: ../../Data/LUAD/2096-Lungcancer/matrix.mtx


,0
0,BT1238_AAATCAACTGCCTC
1,BT1238_AACATTGACCTAAG
2,BT1238_AACCAGTGCTTAGG
3,BT1238_AACCTACTCGCTAA
4,BT1238_AACTCTTGCTGTAG
...,...
93570,scrBT1432_TTTGGTTCATTCTCAT
93571,scrBT1432_TTTGGTTGTTGGTGGA
93572,scrBT1432_TTTGTCACACATGTGT
93573,scrBT1432_TTTGTCAGTACGAAAT


,0,1
0,RP11-34P13.3,RP11-34P13.3
1,FAM138A,FAM138A
2,OR4F5,OR4F5
3,RP11-34P13.7,RP11-34P13.7
4,RP11-34P13.8,RP11-34P13.8
...,...,...
33689,AC233755.2,AC233755.2
33690,AC233755.1,AC233755.1
33691,AC240274.1,AC240274.1
33692,AC213203.1,AC213203.1


,RP11-34P13.3,FAM138A,OR4F5,RP11-34P13.7,RP11-34P13.8,RP11-34P13.14,RP11-34P13.9,FO538757.3,FO538757.2,AP006222.2,...,AC007325.2,BX072566.1,AL354822.1,AC023491.2,AC004556.1,AC233755.2,AC233755.1,AC240274.1,AC213203.1,FAM231B
BT1238_AAATCAACTGCCTC,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1238_AACATTGACCTAAG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1238_AACCAGTGCTTAGG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1238_AACCTACTCGCTAA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
BT1238_AACTCTTGCTGTAG,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scrBT1432_TTTGGTTCATTCTCAT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrBT1432_TTTGGTTGTTGGTGGA,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrBT1432_TTTGTCACACATGTGT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
scrBT1432_TTTGTCAGTACGAAAT,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Running doublet detection on 93575 cells...


/home/wang3712/.local/lib/python3.8/site-packages/scanpy/preprocessing/_normalization.py:170: UserWarning: Received a view of an AnnData. Making a copy.
  view_to_actual(adata)


Automatically set threshold at doublet score = 0.86
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 0.2%
Overall doublet rate:
	Expected   = 6.0%
	Estimated  = 0.5%
  Detected 1 doublets (0.0%)
  After doublet removal: 93574 cells
Finished processing 2096-Lungcancer. Shape: (91192, 33694)


In [48]:
ad.obs['PatientNumber'] = ad.obs['PatientNumber'].astype(str)

In [49]:
ad = filter_and_recompute(adata=ad, 
                          celltype_col='CellType', 
                          celltypes_to_keep=['Cancer'],
                          further_pre=True)
ad

Original shape: (91192, 33694)
Filtered shape: (11085, 33694)
Recalculated PCA and UMAP.


AnnData object with n_obs × n_vars = 11085 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [53]:
patient_cell_number = pd.read_csv("../../Data/LUAD/2096-Lungcancer/2097-Lungcancer_metadata.csv")['PatientNumber'].value_counts()
patient_cell_number = patient_cell_number.to_dict()
patient_cell_number

{5: 21167, 4: 17272, 3: 16791, 7: 16359, 8: 10765, 6: 6949, 2: 2799, 1: 1473}

In [54]:
patient_seuqncing_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Sequencing_quality_S2.txt', sep='\t')
patient_seuqncing_meta_df = patient_seuqncing_meta_df[patient_seuqncing_meta_df['Cancer type'] == 'LC']
patient_seuqncing_meta_df

,Patient number,Cancer type,10X version,Cells,Sample type,Tumour site,UMIs,Saturation (%),Reads
0,LC_1,LC,3' V1,175,Tumour,Core,799729,97.0,52617854
1,LC_1,LC,3' V1,251,Tumour,Core,1268737,96.6,67690218
2,LC_1,LC,3' V1,98,Tumour,Middle,616179,98.1,80569121
3,LC_1,LC,3' V1,302,Tumour,Middle,1489983,95.9,70372636
4,LC_1,LC,3' V1,294,Tumour,Border,1343488,96.2,62162902
5,LC_1,LC,3' V1,353,Tumour,Border,1657136,95.7,62589463
6,LC_2,LC,3' V1,630,Tumour,Middle,3000702,93.2,72138927
7,LC_2,LC,3' V1,31,Tumour,Middle,197373,99.1,62127483
8,LC_2,LC,3' V1,122,Tumour,Border,808401,97.4,55884196
9,LC_2,LC,3' V1,356,Normal,na,1448126,96.9,86230103


In [55]:
# Group by 'Patient number' and sum the 'Cells' column
cells_to_lc_label = patient_seuqncing_meta_df.groupby("Patient number")["Cells"].sum().to_dict()

# Flip the dict so it's {cell_sum: patient_id}
cells_to_lc_label = {v: k for k, v in cells_to_lc_label.items()}
cells_to_lc_label


{1473: 'LC_1',
 2799: 'LC_2',
 16791: 'LC_3',
 17272: 'LC_4',
 21167: 'LC_5',
 6949: 'LC_6',
 16359: 'LC_7',
 7615: 'LC_8'}

In [56]:
patient_number_to_LC_id = dict()
for patient_number in patient_cell_number.keys():
    # print(patient_number)
    try:
        patient_number_to_LC_id[str(patient_number)] = cells_to_lc_label[patient_cell_number[patient_number]]
    except:
        print(patient_number)
patient_number_to_LC_id

8


{'5': 'LC_5',
 '4': 'LC_4',
 '3': 'LC_3',
 '7': 'LC_7',
 '6': 'LC_6',
 '2': 'LC_2',
 '1': 'LC_1'}

In [57]:
patient_number_to_LC_id['8'] = 'LC_8'

In [58]:
ad.obs['BC_PatientID'] = ad.obs['PatientNumber'].map(patient_number_to_LC_id)
ad.obs

,Cell,nGene,nUMI,CellFromTumor,PatientNumber,TumorType,TumorSite,CellType,doublet_score,predicted_doublet,n_genes_by_counts,total_counts,total_counts_mt,pct_counts_mt,Project_ID,Primary_or_Metastatic,BC_PatientID
BT1238_AAATCAACTGCCTC,BT1238_AAATCAACTGCCTC,897,3227,True,1,Lung,I,Cancer,0.114859,False,897,3227.0,75.0,2.324140,2096-Lungcancer,Primary,LC_1
BT1238_AACATTGACCTAAG,BT1238_AACATTGACCTAAG,509,731,True,1,Lung,I,Cancer,0.106954,False,509,731.0,1.0,0.136799,2096-Lungcancer,Primary,LC_1
BT1238_AACTTGCTACTCTT,BT1238_AACTTGCTACTCTT,2351,11433,True,1,Lung,I,Cancer,0.084894,False,2351,11433.0,125.0,1.093326,2096-Lungcancer,Primary,LC_1
BT1238_ACCCAAGATAGAGA,BT1238_ACCCAAGATAGAGA,943,2270,True,1,Lung,I,Cancer,0.098737,False,943,2270.0,68.0,2.995595,2096-Lungcancer,Primary,LC_1
BT1238_ACTTAAGATGCAAC,BT1238_ACTTAAGATGCAAC,3255,20283,True,1,Lung,I,Cancer,0.079009,False,3255,20283.0,112.0,0.552187,2096-Lungcancer,Primary,LC_1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
scrBT1432_TTCGGTCCAGCTGTGC,scrBT1432_TTCGGTCCAGCTGTGC,611,1761,True,8,Lung,I,Cancer,0.050895,False,611,1761.0,23.0,1.306076,2096-Lungcancer,Primary,LC_8
scrBT1432_TTCTCAATCTTCGAGA,scrBT1432_TTCTCAATCTTCGAGA,674,1578,True,8,Lung,I,Cancer,0.051801,False,674,1578.0,35.0,2.217997,2096-Lungcancer,Primary,LC_8
scrBT1432_TTGAACGAGAATGTTG,scrBT1432_TTGAACGAGAATGTTG,359,830,True,8,Lung,I,Cancer,0.070181,False,359,830.0,26.0,3.132530,2096-Lungcancer,Primary,LC_8
scrBT1432_TTGGAACAGACTAAGT,scrBT1432_TTGGAACAGACTAAGT,979,2809,True,8,Lung,I,Cancer,0.095504,False,979,2809.0,29.0,1.032396,2096-Lungcancer,Primary,LC_8


In [59]:
patient_meta_df = pd.read_csv('../../Data/BRCA/2102-Breastcancer/Patient_metadata_S1.txt', sep='\t')
patient_meta_df = patient_meta_df[patient_meta_df['Tumor_type'] == 'LC']
meta_subset = patient_meta_df
meta_subset

,Patient_number,Tumor_type,Gender,Age_range,Stage,TNM,Pathological_subtype,Molecular_status
0,LC_1,LC,Female,70-75,IIA,pT2bN0M0,Squamous cell carcinoma,NaN
1,LC_2,LC,Male,86-90,IB,pT2bN0M0,Squamous cell carcinoma,NaN
2,LC_3,LC,Male,66-70,IIIB,pT4N2M0,Adenocarcinoma,NaN
3,LC_4,LC,Female,60-65,IIB,pT2aN1M0,Adenocarcinoma,NaN
4,LC_5,LC,Male,60-65,IA3,pT1cN0M0,Large cell carcinoma,NaN
5,LC_6,LC,Male,60-65,IIIA,pT4N1M0,Adenocarcinoma,NaN
6,LC_7,LC,Male,60-65,IB,pT2aN0M0,Squamous cell carcinoma,NaN
7,LC_8,LC,Female,50-55,IIB,pT3N0M0,Pleiomorphic carcinoma,NaN


In [60]:
ad.obs = ad.obs.merge(meta_subset, left_on='BC_PatientID', right_on='Patient_number', how='left')
ad.obs

,Cell,nGene,nUMI,CellFromTumor,PatientNumber,TumorType,TumorSite,CellType,doublet_score,predicted_doublet,...,Primary_or_Metastatic,BC_PatientID,Patient_number,Tumor_type,Gender,Age_range,Stage,TNM,Pathological_subtype,Molecular_status
0,BT1238_AAATCAACTGCCTC,897,3227,True,1,Lung,I,Cancer,0.114859,False,...,Primary,LC_1,LC_1,LC,Female,70-75,IIA,pT2bN0M0,Squamous cell carcinoma,NaN
1,BT1238_AACATTGACCTAAG,509,731,True,1,Lung,I,Cancer,0.106954,False,...,Primary,LC_1,LC_1,LC,Female,70-75,IIA,pT2bN0M0,Squamous cell carcinoma,NaN
2,BT1238_AACTTGCTACTCTT,2351,11433,True,1,Lung,I,Cancer,0.084894,False,...,Primary,LC_1,LC_1,LC,Female,70-75,IIA,pT2bN0M0,Squamous cell carcinoma,NaN
3,BT1238_ACCCAAGATAGAGA,943,2270,True,1,Lung,I,Cancer,0.098737,False,...,Primary,LC_1,LC_1,LC,Female,70-75,IIA,pT2bN0M0,Squamous cell carcinoma,NaN
4,BT1238_ACTTAAGATGCAAC,3255,20283,True,1,Lung,I,Cancer,0.079009,False,...,Primary,LC_1,LC_1,LC,Female,70-75,IIA,pT2bN0M0,Squamous cell carcinoma,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11080,scrBT1432_TTCGGTCCAGCTGTGC,611,1761,True,8,Lung,I,Cancer,0.050895,False,...,Primary,LC_8,LC_8,LC,Female,50-55,IIB,pT3N0M0,Pleiomorphic carcinoma,NaN
11081,scrBT1432_TTCTCAATCTTCGAGA,674,1578,True,8,Lung,I,Cancer,0.051801,False,...,Primary,LC_8,LC_8,LC,Female,50-55,IIB,pT3N0M0,Pleiomorphic carcinoma,NaN
11082,scrBT1432_TTGAACGAGAATGTTG,359,830,True,8,Lung,I,Cancer,0.070181,False,...,Primary,LC_8,LC_8,LC,Female,50-55,IIB,pT3N0M0,Pleiomorphic carcinoma,NaN
11083,scrBT1432_TTGGAACAGACTAAGT,979,2809,True,8,Lung,I,Cancer,0.095504,False,...,Primary,LC_8,LC_8,LC,Female,50-55,IIB,pT3N0M0,Pleiomorphic carcinoma,NaN


In [61]:
ad.obs = ad.obs.drop(columns=['Molecular_status'])


In [62]:
ad.obs['Final_cancer_type'] = 'Lung Cancer'
ad.obs['Final_histological_subtype'] = ad.obs.Pathological_subtype
ad.obs['Final_molecular_subtype'] = 'LUCA: Unspecified'
ad.obs['Final_tissue'] = 'Lung'
ad.obs['Final_sample_id'] = ad.obs['BC_PatientID']

In [63]:
ad

AnnData object with n_obs × n_vars = 11085 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'BC_PatientID', 'Patient_number', 'Tumor_type', 'Gender', 'Age_range', 'Stage', 'TNM', 'Pathological_subtype', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue', 'Final_sample_id'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'TumorType_colors', 'PatientNumber_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [64]:
# add more clinical information
ad.obs['Final_patient_age'] = ad.obs['Age_range']
ad.obs['Final_patient_stage'] = ad.obs['TNM']
ad.obs['Final_patient_treatment'] = 'Naïve'

In [65]:
ad.raw.shape

(11085, 33694)

In [68]:
ad

AnnData object with n_obs × n_vars = 11085 × 33694
    obs: 'Cell', 'nGene', 'nUMI', 'CellFromTumor', 'PatientNumber', 'TumorType', 'TumorSite', 'CellType', 'doublet_score', 'predicted_doublet', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'Project_ID', 'Primary_or_Metastatic', 'BC_PatientID', 'Patient_number', 'Tumor_type', 'Gender', 'Age_range', 'Stage', 'TNM', 'Pathological_subtype', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue', 'Final_sample_id', 'Final_patient_age', 'Final_patient_stage', 'Final_patient_treatment'
    var: 'mt', 'n_cells_by_counts', 'mean_counts', 'pct_dropout_by_counts', 'total_counts', 'mean', 'std'
    uns: 'scrublet', 'log1p', 'pca', 'neighbors', 'umap', 'TumorType_colors', 'PatientNumber_colors'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    obsp: 'distances', 'connectivities'

In [67]:
ad.write_h5ad('../../Data/Cancer_cell_data_reprocessed//2096-Lungcancer.LUCA.h5ad', compression='gzip')

## Integrate the data

In [ ]:
data_dir = './Data/Cancer_cell_data/'
all_h5_files = os.listdir(data_dir)
all_h5_files.sort()

all_h5_files

In [ ]:
from collections import defaultdict

cancer_ad_list = []

for h5 in all_h5_files:
    if 'ntegrated' in h5 or 'LUCA' not in h5:
        continue

    print(h5)
    # continue
    ad = sc.read_h5ad(data_dir + h5)
    
    # display(tmp_ad.obs)
    if '2096-Lungcancer' in h5:
        ad.obs_names = ad.obs['Cell']
    elif 'resolution_single' in h5:
        ad.obs_names = ad.obs.index
    elif '_SCLC' in h5:
        ad.obs_names = ad.obs.index

    # Fix .var_names
    if ad.var_names[0].startswith('ENSG'):
        new_names = [i.split('_')[0] for i in ad.var.feature_name]
    elif 'ENSG' in ad.var_names[0]:
        new_names = [i.split('_')[0] for i in ad.var_names]
    else:
        new_names = list(ad.var_names)

    # Assign new names
    ad.var_names = new_names
    ad.var_names_make_unique()

    # Fix raw.var names
    if ad.raw is not None:
        ad.raw._var.index = pd.Index(new_names).astype(str)
        # Ensure uniqueness
        seen = defaultdict(int)
        unique_names = []
        for name in ad.raw._var.index:
            if seen[name]:
                unique_names.append(f"{name}_{seen[name]}")
            else:
                unique_names.append(name)
            seen[name] += 1
        ad.raw._var.index = pd.Index(unique_names)

    # Clean obs + var
    # ad.obs = ad.obs.reset_index(drop=True)
    ad.obs_names_make_unique()
    ad.var_names_make_unique()

    cancer_ad_list.append(ad)
    display(ad.to_df())
    display(ad.raw.to_adata().to_df())
    print(ad.raw.to_adata().to_df().max(axis=1))


In [ ]:
for ad in cancer_ad_list:
    print(ad.raw.shape)

In [ ]:
memory_usgae()

In [ ]:
combined_ad = ann.concat(cancer_ad_list, join="inner", axis=0)
combined_ad

In [ ]:
combined_ad.raw.shape

In [ ]:
combined_ad = reprocess_all(combined_ad)

In [ ]:
combined_ad

In [ ]:
combined_ad.obs["Final_histological_subtype"].value_counts()

In [ ]:
combined_ad.obs["Final_histological_subtype_backup"] = combined_ad.obs["Final_histological_subtype"].copy()


In [ ]:
def unify_histological_subtype(value):
    value = str(value).strip().lower()
    if value in {"luad", "adenocarcinoma"}:
        return "LUCA: Lung adenocarcinoma"
    elif value in {"lusc", "squamous cell carcinoma"}:
        return "LUCA: Lung squamous carcinoma"
    elif value == "nsclc":
        return "LUCA: NSCLC"
    elif value == "sclc":
        return "LUCA: Small cell lung cancer"
    elif value == "large cell carcinoma":
        return "LUCA: Large cell carcinoma"
    elif value == "pleiomorphic carcinoma":
        return "LUCA: Pleiomorphic carcinoma"
    else:
        return "LUCA: Unspecified"

combined_ad.obs["Final_histological_subtype"] = combined_ad.obs["Final_histological_subtype_backup"].apply(unify_histological_subtype)
combined_ad.obs['Final_histological_subtype'].value_counts()

In [ ]:
combined_ad.obs['Final_molecular_subtype'].value_counts()

In [ ]:
def unify_tissue(tissue):
    tissue = str(tissue).strip().lower()
    if "adrenal" in tissue:
        return "Adrenal"
    # add more rules here as needed
    return tissue.capitalize()

combined_ad.obs["Final_tissue"] = combined_ad.obs["Final_tissue"].apply(unify_tissue)

In [ ]:
combined_ad

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

### Harmony integration

In [ ]:
combined_ad

In [ ]:
Z = harmonize(combined_ad.obsm['X_pca'], combined_ad.obs, batch_key = ['Project_ID'])


In [ ]:
combined_ad.obsm['X_pca_harmony'] = Z


In [ ]:
sc.pp.neighbors(combined_ad, n_neighbors=15, use_rep='X_pca_harmony')
sc.tl.umap(combined_ad)

In [ ]:
for obs in ['Project_ID', 'Primary_or_Metastatic', 'Final_cancer_type', 'Final_histological_subtype', 'Final_molecular_subtype', 'Final_tissue']:
    sc.pl.umap(combined_ad, color=obs)

In [ ]:
combined_ad.obs["Final_patient_age_backup"] = combined_ad.obs["Final_patient_age"]

def clean_patient_age(age):
    if pd.isna(age):
        return np.nan
    age = str(age).strip()
    if age.lower() == "unknown":
        return np.nan
    elif "-" in age:
        # Convert age ranges like '46-50' to their midpoint
        parts = age.split("-")
        try:
            return int((int(parts[0]) + int(parts[1])) / 2)
        except:
            return np.nan
    else:
        try:
            return int(age)
        except:
            return np.nan

# Apply cleaning
combined_ad.obs["Final_patient_age"] = combined_ad.obs["Final_patient_age_backup"].apply(clean_patient_age)
combined_ad.obs["Final_patient_age_backup"] =combined_ad.obs["Final_patient_age_backup"].astype(str)

In [ ]:

combined_ad.write_h5ad('./Data/Cancer_cell_data/LUCA_integrated.harmony.h5ad', compression='gzip')
